## Data Loading

Data Description:  
- 4000 grayscale images of 28x28, vectorized into 784-dimensional feature vectors  
- 2 independent training/test sets, each containing 2000 samples  
- Label range: 0~9 (10 classes of handwritten digits)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

os.makedirs('result', exist_ok=True)

# Select device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device used: {device}')
print(f'PyTorch version: {torch.__version__}')

# Set random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
DATA_DIR = 'digits4000_txt'

X_all    = np.loadtxt(os.path.join(DATA_DIR, 'digits4000_digits_vec.txt'),    delimiter='\t')
y_all    = np.loadtxt(os.path.join(DATA_DIR, 'digits4000_digits_labels.txt'), dtype=int)
trainset = np.loadtxt(os.path.join(DATA_DIR, 'digits4000_trainset.txt'),      delimiter='\t', dtype=int)
testset  = np.loadtxt(os.path.join(DATA_DIR, 'digits4000_testset.txt'),       delimiter='\t', dtype=int)

# Normalize to [0, 1]
X_norm = (X_all / 255.0).astype(np.float32)

# 2 independent experiment indices (0-based)
exp_idx = {
    1: {'train': trainset[:, 0] - 1, 'test': testset[:, 0] - 1},
    2: {'train': trainset[:, 1] - 1, 'test': testset[:, 1] - 1},
}

print(f'X_all: {X_all.shape}, y_all: {y_all.shape}')
for exp in [1, 2]:
    tr_idx = exp_idx[exp]['train']
    te_idx = exp_idx[exp]['test']
    print(f'Experiment {exp}: Train={len(tr_idx)}, Test={len(te_idx)}, '
          f'Class distribution={dict(zip(*np.unique(y_all[tr_idx], return_counts=True)))}')
print('Data loaded.')

# Handwritten Digit Classification - Deep Learning Methods

Implementation of deep learning classification methods, including:
1. **Data Augmentation**: Translation, Rotation, Scaling
2. **Multi-Layer Perceptron (MLP)**: 2-3 hidden layers + ReLU + Dropout + BatchNorm
3. **Convolutional Neural Network (CNN)**: LeNet-5 style, Conv2D → MaxPool → Dropout → Dense

2 independent experiments, hyperparameter tuning based on the training set

## 1. Data Augmentation

Perform the following augmentation operations on 28x28 grayscale images:
- **Random Translation**: Up to approx. 3 pixels in all directions (~10%)
- **Random Rotation**: ±15 degrees
- **Random Scaling**: 0.9~1.1x

In [ ]:
class DigitDataset(Dataset):
    """Handwritten digit Dataset supporting data augmentation (random affine transformations)"""

    def __init__(self, X, y, augment=False):
        self.X = X          # (N, 784) float32
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment
        self.aug_transform = transforms.Compose([
            transforms.RandomAffine(
                degrees=15,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1),
                fill=0
            )
        ])

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        img = torch.tensor(self.X[idx], dtype=torch.float32).view(1, 28, 28)
        if self.augment:
            img = self.aug_transform(img)
        return img, self.y[idx]


# Visualize data augmentation effects
exp1_train_idx = exp_idx[1]['train']
X_train1 = X_norm[exp1_train_idx]
y_train1  = y_all[exp1_train_idx]

sample_idx = [int(np.where(y_train1 == d)[0][0]) for d in range(5)]

np.random.seed(123)
torch.manual_seed(123)

fig, axes = plt.subplots(5, 6, figsize=(14, 12))
aug_ds  = DigitDataset(X_train1, y_train1, augment=True)
orig_ds = DigitDataset(X_train1, y_train1, augment=False)

for row, si in enumerate(sample_idx):
    orig_img, lbl = orig_ds[si]
    axes[row, 0].imshow(orig_img.squeeze().numpy(), cmap='gray')
    axes[row, 0].set_title(f'Original (label:{lbl.item()})', fontsize=8)
    axes[row, 0].axis('off')
    for col in range(1, 6):
        aug_img, _ = aug_ds[si]
        axes[row, col].imshow(aug_img.squeeze().numpy(), cmap='gray')
        axes[row, col].set_title(f'Aug {col}', fontsize=8)
        axes[row, col].axis('off')

plt.suptitle('Data Augmentation Examples\n(Random shift ±10%, rotation ±15°, scale 0.9–1.1×)', fontsize=11)
plt.tight_layout()
plt.savefig('result/12_data_augmentation.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: result/12_data_augmentation.png')

## 2. Model Definition

### 2.1 MLP (Multi-Layer Perceptron)
Architecture: 784 → 512 → BN → ReLU → Dropout → 256 → BN → ReLU → Dropout → 128 → BN → ReLU → Dropout → 10

In [ ]:
class MLP(nn.Module):
    """Multi-Layer Perceptron: 3 hidden layers + ReLU + BatchNorm + Dropout"""

    def __init__(self, input_dim=784, hidden_dims=None, num_classes=10, dropout_rate=0.4):
        super(MLP, self).__init__()
        if hidden_dims is None:
            hidden_dims = [512, 256, 128]
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(in_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_rate)
            ])
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
        return self.network(x)


mlp_model = MLP()
print('=== MLP Model Architecture ===')
print(mlp_model)
total_params = sum(p.numel() for p in mlp_model.parameters())
print(f'\nTotal parameters: {total_params:,}')

### 2.2 CNN (Convolutional Neural Network)
LeNet-5 Style: `Conv2d(5×5)` → `BN` → `ReLU` → `MaxPool(2×2)` → … → `Flatten` → `FC` × 2 → `Dropout` → Output

In [ ]:
class CNN(nn.Module):
    """
    LeNet-5 Style CNN:
    Conv2d(1,6,5,pad=2) -> BN -> ReLU -> MaxPool(2) ->
    Conv2d(6,16,5) -> BN -> ReLU -> MaxPool(2) ->
    Flatten -> FC(400,120) -> ReLU -> Dropout ->
    FC(120,84) -> ReLU -> Dropout -> FC(84,10)

    Input: (B, 1, 28, 28)
    """

    def __init__(self, num_classes=10, dropout_rate=0.3):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(6),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.fc = nn.Sequential(
            nn.Linear(400, 120),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(120, 84),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(84, num_classes)
        )

    def forward(self, x):
        if x.dim() == 2:
            x = x.view(x.size(0), 1, 28, 28)
        elif x.dim() == 3:
            x = x.unsqueeze(1)
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


cnn_model = CNN()
print('=== CNN Model Architecture ===')
print(cnn_model)
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f'\nTotal parameters: {total_params:,}')

# Forward pass validation
dummy = torch.randn(4, 1, 28, 28)
out_cnn = cnn_model(dummy)
print(f'CNN: Input {tuple(dummy.shape)} -> Output {tuple(out_cnn.shape)}')

dummy_flat = dummy.view(4, -1)
out_mlp = mlp_model(dummy_flat)
print(f'MLP: Input {tuple(dummy_flat.shape)} -> Output {tuple(out_mlp.shape)}')

## 3. Model Training and Evaluation Pipeline

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        _, predicted = outputs.max(1)
        correct += predicted.eq(y_batch).sum().item()
        total += len(y_batch)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        total_loss += loss.item() * len(y_batch)
        _, predicted = outputs.max(1)
        correct += predicted.eq(y_batch).sum().item()
        total += len(y_batch)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())
    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


def train_model(model_class, model_kwargs, X_train, y_train, X_test, y_test,
                augment=False, epochs=50, batch_size=64, lr=1e-3,
                weight_decay=1e-4, patience=15, model_name='model',
                val_ratio=0.1):
    """
    Complete training pipeline (strictly no tuning on the test set):
    - Split val_ratio from the training set for validation, used only for early stopping
    - Adam + ReduceLROnPlateau + EarlyStopping
    """
    full_ds = DigitDataset(X_train, y_train, augment=augment)
    n_val = int(len(full_ds) * val_ratio)
    n_tr  = len(full_ds) - n_val
    torch.manual_seed(SEED)
    train_ds, val_ds = random_split(full_ds, [n_tr, n_val])
    
    # No augmentation for the validation set
    val_ds_no_aug = DigitDataset(X_train[list(val_ds.indices)], y_train[list(val_ds.indices)], augment=False)
    test_ds = DigitDataset(X_test, y_test, augment=False)

    train_loader = DataLoader(train_ds,      batch_size=batch_size, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds_no_aug, batch_size=256,        shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,       batch_size=256,        shuffle=False, num_workers=0)

    torch.manual_seed(SEED)
    model = model_class(**model_kwargs).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_state = None
    no_improve = 0

    print(f'  Training {model_name} | epochs={epochs}, batch={batch_size}, lr={lr}')
    print(f'  Train: {n_tr}, Validation: {n_val}, Test: {len(test_ds)}')

    t_start = time.time()
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc, _, _ = eval_epoch(model, val_loader, criterion)
        scheduler.step(vl_loss)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if ep % 10 == 0 or ep == 1:
            print(f'  Epoch {ep:3d}/{epochs} | '
                  f'train_loss={tr_loss:.4f} acc={tr_acc:.4f} | '
                  f'val_loss={vl_loss:.4f} acc={vl_acc:.4f} | '
                  f'lr={optimizer.param_groups[0]["lr"]:.2e}')

        if no_improve >= patience:
            print(f'  EarlyStopping at epoch {ep}')
            break

    model.load_state_dict(best_state)
    test_loss, test_acc, y_pred, y_true = eval_epoch(model, test_loader, criterion)
    elapsed = time.time() - t_start
    print(f'  Test Accuracy: {test_acc:.4f} | Total Time: {elapsed:.1f}s')

    return model, history, test_acc, y_pred, y_true


print('Training functions defined.')

## 4. Training Results Logging

In [ ]:
dl_results = {}  # Store all deep learning method results

def plot_training_history(history, title, save_path):
    """Plot training and validation loss/accuracy curves"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history['train_loss']) + 1)

    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    ax1.plot(epochs, history['val_loss'],   'r-', label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} - Loss Curve')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    ax2.plot(epochs, history['val_acc'],   'r-', label='Val Acc')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{title} - Accuracy Curve')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'Saved: {save_path}')

print('dl_results initialized, plot_training_history function defined.')

## 5. MLP Training (Without Data Augmentation)

In [ ]:
print('=' * 60)
print('  MLP (3 hidden layers, no data augmentation)')
print('=' * 60)

mlp_accs = []
mlp_histories = []

for exp in [1, 2]:
    print(f'\n--- Experiment {exp} ---')
    tr_idx = exp_idx[exp]['train']
    te_idx = exp_idx[exp]['test']
    X_tr = X_norm[tr_idx]
    y_tr = y_all[tr_idx]
    X_te = X_norm[te_idx]
    y_te = y_all[te_idx]

    model, hist, test_acc, y_pred, y_true = train_model(
        MLP,
        {'hidden_dims': [512, 256, 128], 'dropout_rate': 0.4},
        X_tr, y_tr, X_te, y_te,
        augment=False,
        epochs=80,
        batch_size=64,
        lr=1e-3,
        weight_decay=1e-4,
        patience=15,
        model_name=f'MLP_exp{exp}'
    )
    mlp_accs.append(test_acc)
    mlp_histories.append(hist)

    plot_training_history(hist, f'MLP Exp{exp} (No Aug)',
                          f'result/13_mlp_exp{exp}_history.png')

dl_results['MLP (3-layer, no aug)'] = {
    'exp1': mlp_accs[0], 'exp2': mlp_accs[1],
    'mean': float(np.mean(mlp_accs)), 'std': float(np.std(mlp_accs))
}
print(f'\nMLP (no aug) mean accuracy: {np.mean(mlp_accs):.4f} ± {np.std(mlp_accs):.4f}')

## 6. MLP Training (With Data Augmentation)

In [ ]:
print('=' * 60)
print('  MLP (3 hidden layers + Data Augmentation)')
print('=' * 60)

mlp_aug_accs = []

for exp in [1, 2]:
    print(f'\n--- Experiment {exp} ---')
    tr_idx = exp_idx[exp]['train']
    te_idx = exp_idx[exp]['test']
    
    model_aug, hist_aug, test_acc, y_pred, y_true = train_model(
        MLP,
        {'hidden_dims': [512, 256, 128], 'dropout_rate': 0.4},
        X_norm[tr_idx], y_all[tr_idx],
        X_norm[te_idx], y_all[te_idx],
        augment=True,  # Enable data augmentation
        epochs=100,
        batch_size=64,
        lr=1e-3,
        weight_decay=1e-4,
        patience=20,
        model_name=f'MLP_Aug_exp{exp}'
    )
    mlp_aug_accs.append(test_acc)
    plot_training_history(hist_aug, f'MLP Exp{exp} (w/ Aug)',
                           f'result/14_mlp_aug_exp{exp}_history.png')

dl_results['MLP (3-layer, w/ aug)'] = {
    'exp1': mlp_aug_accs[0], 'exp2': mlp_aug_accs[1],
    'mean': np.mean(mlp_aug_accs), 'std': np.std(mlp_aug_accs)
}
print(f'\nMLP (w/ aug) mean accuracy: {np.mean(mlp_aug_accs):.4f} ± {np.std(mlp_aug_accs):.4f}')

## 7. MLP Network Depth Comparison Experiment

In [ ]:
print('=' * 60)
print('  MLP Network Depth Comparison (1/2/3 Hidden Layers)')
print('=' * 60)

depth_configs = {
    '1-layer(512)':     [512],
    '2-layer(512→256)': [512, 256],
    '3-layer(512→256→128)': [512, 256, 128]
}

depth_accs = {}
for config_name, hidden_dims in depth_configs.items():
    accs = []
    for exp in [1, 2]:
        tr_idx = exp_idx[exp]['train']
        te_idx = exp_idx[exp]['test']
        _, _, test_acc, _, _ = train_model(
            MLP,
            {'hidden_dims': hidden_dims, 'dropout_rate': 0.4},
            X_norm[tr_idx], y_all[tr_idx],
            X_norm[te_idx], y_all[te_idx],
            augment=False, epochs=60, batch_size=64,
            lr=1e-3, patience=12,
            model_name=f'MLP_{config_name}_exp{exp}'
        )
        accs.append(test_acc)
    depth_accs[config_name] = {'exp1': accs[0], 'exp2': accs[1], 'mean': np.mean(accs)}
    print(f'  {config_name}: Exp 1={accs[0]:.4f}, Exp 2={accs[1]:.4f}, Mean={np.mean(accs):.4f}')

# Visualize depth comparison
fig, ax = plt.subplots(figsize=(9, 4))
names = list(depth_configs.keys())
e1s = [depth_accs[n]['exp1'] for n in names]
e2s = [depth_accs[n]['exp2'] for n in names]
means = [depth_accs[n]['mean'] for n in names]
x = np.arange(len(names))
ax.bar(x - 0.25, e1s, 0.25, label='Exp 1', color='steelblue', alpha=0.8)
ax.bar(x, e2s, 0.25, label='Exp 2', color='coral', alpha=0.8)
ax.bar(x + 0.25, means, 0.25, label='Mean', color='green', alpha=0.8)
ax.axhline(y=0.9160, color='red', linestyle='--', label='1-NN Baseline')
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel('Test Accuracy')
ax.set_title('MLP: Effect of Network Depth')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0.85, 1.0)
plt.tight_layout()
plt.savefig('result/15_mlp_depth_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: result/15_mlp_depth_comparison.png')

## 8. CNN Training (Without Data Augmentation)

In [ ]:
print('=' * 60)
print('  CNN (LeNet-5 Style, No Data Augmentation)')
print('  Input: 784D -> Reshape to 28x28 -> Conv2D -> MaxPool -> ...')
print('=' * 60)

cnn_accs = []

for exp in [1, 2]:
    print(f'\n--- Experiment {exp} ---')
    tr_idx = exp_idx[exp]['train']
    te_idx = exp_idx[exp]['test']
    
    model_cnn, hist_cnn, test_acc, y_pred, y_true = train_model(
        CNN,
        {'dropout_rate': 0.3},
        X_norm[tr_idx], y_all[tr_idx],
        X_norm[te_idx], y_all[te_idx],
        augment=False,
        epochs=80,
        batch_size=64,
        lr=1e-3,
        weight_decay=1e-4,
        patience=15,
        model_name=f'CNN_exp{exp}'
    )
    cnn_accs.append(test_acc)
    plot_training_history(hist_cnn, f'CNN Exp{exp} (No Aug)',
                           f'result/16_cnn_exp{exp}_history.png')

dl_results['CNN (LeNet-5, no aug)'] = {
    'exp1': cnn_accs[0], 'exp2': cnn_accs[1],
    'mean': np.mean(cnn_accs), 'std': np.std(cnn_accs)
}
print(f'\nCNN (no aug) mean accuracy: {np.mean(cnn_accs):.4f} ± {np.std(cnn_accs):.4f}')

## 9. CNN Training (With Data Augmentation)

In [ ]:
print('=' * 60)
print('  CNN (LeNet-5 Style + Data Augmentation)')
print('=' * 60)

cnn_aug_accs = []
cnn_aug_preds = []

for exp in [1, 2]:
    print(f'\n--- Experiment {exp} ---')
    tr_idx = exp_idx[exp]['train']
    te_idx = exp_idx[exp]['test']
    
    model_cnn_aug, hist_cnn_aug, test_acc, y_pred, y_true = train_model(
        CNN,
        {'dropout_rate': 0.3},
        X_norm[tr_idx], y_all[tr_idx],
        X_norm[te_idx], y_all[te_idx],
        augment=True,  # Enable data augmentation
        epochs=100,
        batch_size=64,
        lr=1e-3,
        weight_decay=1e-4,
        patience=20,
        model_name=f'CNN_Aug_exp{exp}'
    )
    cnn_aug_accs.append(test_acc)
    cnn_aug_preds.append((y_pred, y_true))
    plot_training_history(hist_cnn_aug, f'CNN Exp{exp} (w/ Aug)',
                           f'result/17_cnn_aug_exp{exp}_history.png')

dl_results['CNN (LeNet-5, w/ aug)'] = {
    'exp1': cnn_aug_accs[0], 'exp2': cnn_aug_accs[1],
    'mean': np.mean(cnn_aug_accs), 'std': np.std(cnn_aug_accs)
}
print(f'\nCNN (w/ aug) mean accuracy: {np.mean(cnn_aug_accs):.4f} ± {np.std(cnn_aug_accs):.4f}')

## 10. CNN Dropout Parameter Comparison Experiment

In [ ]:
print('=' * 60)
print('  CNN Dropout Rate Comparison')
print('=' * 60)

dropout_vals = [0.0, 0.2, 0.3, 0.5]
dropout_accs = {}

for dr in dropout_vals:
    accs = []
    for exp in [1, 2]:
        tr_idx = exp_idx[exp]['train']
        te_idx = exp_idx[exp]['test']
        _, _, test_acc, _, _ = train_model(
            CNN, {'dropout_rate': dr},
            X_norm[tr_idx], y_all[tr_idx],
            X_norm[te_idx], y_all[te_idx],
            augment=False, epochs=60, batch_size=64, lr=1e-3, patience=12,
            model_name=f'CNN_dr{dr}_exp{exp}'
        )
        accs.append(test_acc)
    dropout_accs[dr] = {'exp1': accs[0], 'exp2': accs[1], 'mean': np.mean(accs)}
    print(f'  Dropout={dr}: Exp 1={accs[0]:.4f}, Exp 2={accs[1]:.4f}, Mean={np.mean(accs):.4f}')

# Dropout comparison chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dropout_vals, [dropout_accs[d]['exp1'] for d in dropout_vals], 'b-o', label='Exp 1')
ax.plot(dropout_vals, [dropout_accs[d]['exp2'] for d in dropout_vals], 'r-s', label='Exp 2')
ax.plot(dropout_vals, [dropout_accs[d]['mean'] for d in dropout_vals], 'g--^', label='Mean', lw=2)
ax.axhline(y=0.9160, color='orange', linestyle='--', label='1-NN Baseline')
ax.set_xlabel('Dropout Rate')
ax.set_ylabel('Test Accuracy')
ax.set_title('CNN: Effect of Dropout Rate on Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('result/18_cnn_dropout_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: result/18_cnn_dropout_analysis.png')

## 11. CNN Confusion Matrix Analysis

In [ ]:
# Confusion matrix of the best CNN (with augmentation)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, exp_i, (y_pred, y_true) in zip(axes, [1, 2], cnn_aug_preds):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(10))
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_true, y_pred)
    ax.set_title(f'Exp {exp_i}: CNN (LeNet-5+Aug)  Acc={acc:.4f}', fontsize=10)

plt.suptitle('CNN Confusion Matrix Analysis', fontsize=12)
plt.tight_layout()
plt.savefig('result/19_cnn_confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: result/19_cnn_confusion_matrix.png')

## 12. Analysis of Typical Success and Failure Cases

In [ ]:
# Analyze typical error cases in Experiment 1
y_pred_exp1, y_true_exp1 = cnn_aug_preds[0]
te_idx_exp1 = exp_idx[1]['test']
X_test_exp1 = X_norm[te_idx_exp1]

wrong_mask = y_pred_exp1 != y_true_exp1
wrong_indices = np.where(wrong_mask)[0]

print(f'Experiment 1 - CNN (w/ Aug) Error count: {len(wrong_indices)} / {len(y_true_exp1)}')

# Show up to 20 error samples
n_show = min(20, len(wrong_indices))
np.random.seed(42)
show_idx = np.random.choice(wrong_indices, n_show, replace=False)

fig, axes = plt.subplots(4, 5, figsize=(12, 10))
for i, (ax, idx) in enumerate(zip(axes.flatten(), show_idx)):
    img = X_test_exp1[idx].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    true_label = y_true_exp1[idx]
    pred_label = y_pred_exp1[idx]
    ax.set_title(f'True: {true_label}  Pred: {pred_label}',
                 color='red', fontsize=8)
    ax.axis('off')

plt.suptitle('CNN Typical Misclassification Cases (Exp 1)', fontsize=12)
plt.tight_layout()
plt.savefig('result/20_cnn_error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: result/20_cnn_error_analysis.png')

# Most frequent misclassification pairs
from collections import Counter
error_pairs = Counter(zip(y_true_exp1[wrong_mask], y_pred_exp1[wrong_mask]))
print('\nMost common confusion pairs (True -> Pred):')
for (true, pred), count in error_pairs.most_common(10):
    print(f'  Digit {true} -> Digit {pred}: {count} times')

In [ ]:
# ---- Typical Success Cases Analysis (Exp 1) ----
y_pred_exp1, y_true_exp1 = cnn_aug_preds[0]
te_idx1 = exp_idx[1]['test']
X_test1 = X_norm[te_idx1]

correct_mask = (y_pred_exp1 == y_true_exp1)

# Select correct samples with the highest confidence for each class (max 2)
success_samples = []
for digit in range(10):
    cls_mask = correct_mask & (y_true_exp1 == digit)
    idxs = np.where(cls_mask)[0]
    if len(idxs) > 0:
        for si in idxs[:2]:
            success_samples.append((si, y_true_exp1[si], y_pred_exp1[si]))
        if len(success_samples) >= 20:
            break

n_success = min(20, len(success_samples))
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
axes = axes.ravel()
for k, (si, true_lbl, pred_lbl) in enumerate(success_samples[:n_success]):
    img = X_test1[si].reshape(28, 28)
    axes[k].imshow(img, cmap='gray')
    axes[k].set_title(f'True:{true_lbl}  Pred:{pred_lbl}', color='green', fontsize=8)
    axes[k].axis('off')
for k in range(n_success, len(axes)):
    axes[k].axis('off')

plt.suptitle('Typical Correct Classifications (Exp 1, CNN+Aug)\nRepresentative correct samples per class',
             fontsize=11)
plt.tight_layout()
plt.savefig('result/19_success_cases.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: result/19_success_cases.png')
print(f'Displayed {n_success} typical success cases')

## 13. Comprehensive Comparison of All Deep Learning Methods

In [ ]:
print('\n' + '=' * 70)
print('        Summary of Deep Learning Methods Accuracy')
print('=' * 70)
print(f'{"Method":<30} {"Exp 1":>8} {"Exp 2":>8} {"Mean":>8} {"Std":>8}')
print('-' * 70)
print(f'{"1-NN Baseline (Official)":<30} {0.9135:>8.4f} {0.9185:>8.4f} {0.9160:>8.4f} {0.0035:>8.4f}')
for name, r in dl_results.items():
    marker = ' *' if r['mean'] > 0.9160 else '  '
    print(f'{name:<30} {r["exp1"]:>8.4f} {r["exp2"]:>8.4f} {r["mean"]:>8.4f} {r["std"]:>8.4f}{marker}')
print('=' * 70)
print('Note: * indicates performance above 1-NN Baseline (0.9160)')

In [ ]:
# Comprehensive comparison chart for deep learning methods
method_names = list(dl_results.keys())
exp1_accs = [dl_results[m]['exp1'] for m in method_names]
exp2_accs = [dl_results[m]['exp2'] for m in method_names]
mean_accs = [dl_results[m]['mean'] for m in method_names]
std_accs  = [dl_results[m]['std']  for m in method_names]

x = np.arange(len(method_names))
width = 0.27

fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - width, exp1_accs, width, label='Exp 1', color='steelblue', alpha=0.85)
b2 = ax.bar(x, exp2_accs, width, label='Exp 2', color='coral', alpha=0.85)
b3 = ax.bar(x + width, mean_accs, width, label='Mean', color='mediumseagreen', alpha=0.85,
            yerr=std_accs, capsize=4, error_kw={'elinewidth': 1.5})

ax.axhline(y=0.9160, color='red', linestyle='--', linewidth=2, label='1-NN Baseline (0.9160)')

ax.set_xticks(x)
ax.set_xticklabels(method_names, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Test Accuracy')
ax.set_title('Deep Learning Methods Accuracy Comparison (vs 1-NN Baseline)', fontsize=13)
ax.legend(loc='lower right')
ax.set_ylim(0.85, 1.0)
ax.grid(True, axis='y', alpha=0.3)

# Annotate mean values
for bar, mean in zip(b3, mean_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{mean:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('result/21_dl_methods_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: result/21_dl_methods_comparison.png')

## 14. Data Augmentation Effect Comparison (MLP vs CNN)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# MLP: No Aug vs w/ Aug
ax = axes[0]
categories = ['Exp 1', 'Exp 2', 'Mean']
no_aug = [dl_results['MLP (3-layer, no aug)']['exp1'],
           dl_results['MLP (3-layer, no aug)']['exp2'],
           dl_results['MLP (3-layer, no aug)']['mean']]
with_aug = [dl_results['MLP (3-layer, w/ aug)']['exp1'],
             dl_results['MLP (3-layer, w/ aug)']['exp2'],
             dl_results['MLP (3-layer, w/ aug)']['mean']]
x = np.arange(3)
ax.bar(x - 0.2, no_aug, 0.4, label='No Aug', color='steelblue', alpha=0.8)
ax.bar(x + 0.2, with_aug, 0.4, label='w/ Aug', color='orange', alpha=0.8)
ax.axhline(y=0.9160, color='red', linestyle='--', label='1-NN Baseline')
ax.set_title('MLP: Effect of Data Augmentation')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.set_ylim(0.88, 1.0)
ax.grid(True, axis='y', alpha=0.3)

# CNN: No Aug vs w/ Aug
ax = axes[1]
no_aug_cnn = [dl_results['CNN (LeNet-5, no aug)']['exp1'],
               dl_results['CNN (LeNet-5, no aug)']['exp2'],
               dl_results['CNN (LeNet-5, no aug)']['mean']]
with_aug_cnn = [dl_results['CNN (LeNet-5, w/ aug)']['exp1'],
                 dl_results['CNN (LeNet-5, w/ aug)']['exp2'],
                 dl_results['CNN (LeNet-5, w/ aug)']['mean']]
ax.bar(x - 0.2, no_aug_cnn, 0.4, label='No Aug', color='steelblue', alpha=0.8)
ax.bar(x + 0.2, with_aug_cnn, 0.4, label='w/ Aug', color='orange', alpha=0.8)
ax.axhline(y=0.9160, color='red', linestyle='--', label='1-NN Baseline')
ax.set_title('CNN: Effect of Data Augmentation')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel('Test Accuracy')
ax.legend()
ax.set_ylim(0.88, 1.0)
ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Impact of Data Augmentation on Deep Learning Models', fontsize=12)
plt.tight_layout()
plt.savefig('result/22_augmentation_effect_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: result/22_augmentation_effect_comparison.png')